# MSC Fingerprinting — Scaffold vs FC

Replicate the core fingerprinting analysis (Fig 2 Panel A+B style) on the MSC dataset.

- 10 subjects, 10 sessions each (schaefer400-yeo17, regress_36pNS, task-rest)
- For each pair of sessions (s_i, s_j): build N×N similarity matrix, compute success rate, effect size, Idiff
- Compare: FC full, FC th=25%, FC th=1%, Scaffold

In [ ]:
import sys
import sys
sys.path.append("..")
from topofinger import config

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sb
import pandas as pd
import pickle as pk
import os
from collections import OrderedDict
from itertools import combinations
from scipy.stats import pearsonr
from pingouin import compute_effsize

from topofinger.fingerprinting import compute_idiff, compute_effect_size, compute_success_rate
from topofinger.utils import apply_threshold_to_nested_dict
from topofinger.viz import plot_identifiability

%load_ext autoreload
%autoreload 2

In [ ]:
data_dir = config.MSC + '/'
figure_dir = config.FIGURES_RAW + '/MSC/'
os.makedirs(figure_dir, exist_ok=True)

## 1. Load data

In [ ]:
# ── Load FC ──
with open(data_dir + 'fc/whole/whole_corrs.pck', 'rb') as f:
    fc = pk.load(f)

subjects = sorted(fc.keys())
n_sub = len(subjects)
sessions = sorted(fc[subjects[0]].keys())
n_sess = len(sessions)

print(f'Subjects: {n_sub}, Sessions: {n_sess}')
print(f'FC matrix shape: {fc[subjects[0]][sessions[0]].shape}')

In [ ]:
# ── Load scaffolds ──
def load_scaffold_msc(path):
    """Load MSC scaffolds into dict[subject][session] = np.array"""
    fs = sorted(os.listdir(path))
    sc = OrderedDict()
    for f in fs:
        # filename: sub-MSCxx_sN
        parts = f.rsplit('_s', 1)
        sub = parts[0]
        sess = int(parts[1])
        if sub not in sc:
            sc[sub] = OrderedDict()
        sc[sub][sess] = np.load(os.path.join(path, f))
    return sc

sc = load_scaffold_msc(data_dir + 'scaffolds/whole/freq/')
print(f'Scaffold subjects: {len(sc)}')
print(f'Sessions per subject: {[len(sc[s]) for s in sc]}')
print(f'Scaffold shape: {sc[subjects[0]][0].shape}')
print(f'Scaffold density: {np.count_nonzero(sc[subjects[0]][0]) / sc[subjects[0]][0].size * 100:.2f}%')

## 2. Build similarity matrices

For a given pair of sessions (test, retest), build the N×N similarity matrix where
`sim[i,j] = pearsonr(flatten(subject_i_test), flatten(subject_j_retest))`.

With 10 sessions we have $\binom{10}{2} = 45$ session pairs.

In [ ]:
def build_sim_matrix(matrices_dict, sess_test, sess_retest):
    """
    Build N×N similarity matrix for a given test-retest session pair.
    matrices_dict[subject][session] = np.array (regions × regions)
    """
    subs = sorted(matrices_dict.keys())
    n = len(subs)
    
    test_vecs = np.array([np.nan_to_num(matrices_dict[s][sess_test]).flatten() for s in subs])
    retest_vecs = np.array([np.nan_to_num(matrices_dict[s][sess_retest]).flatten() for s in subs])
    
    # Use numpy correlation: corrcoef of stacked gives (2N×2N), we want the NxN block
    full = np.corrcoef(test_vecs, retest_vecs)
    sim = full[:n, n:]  # test vs retest block
    return sim

In [ ]:
# ── Compute similarity matrices for all session pairs ──
session_pairs = list(combinations(sessions, 2))
print(f'Number of session pairs: {len(session_pairs)}')

# Thresholded FC variants
import copy
fc_th25 = apply_threshold_to_nested_dict(fc, 25)
fc_th1 = apply_threshold_to_nested_dict(fc, 1)

conditions = {
    'FC full': fc,
    'FC th=25%': fc_th25,
    'FC th=1%': fc_th1,
    'Scaffold': sc
}

results = {}
for cond_name, cond_data in conditions.items():
    results[cond_name] = {
        'sim_matrices': {},
        'success_rates': [],
        'effect_sizes': [],
        'idiffs': [],
        'i_selfs': [],
        'i_others': []
    }
    for s_test, s_retest in session_pairs:
        sim = build_sim_matrix(cond_data, s_test, s_retest)
        results[cond_name]['sim_matrices'][(s_test, s_retest)] = sim
        
        sr1, sr2 = compute_success_rate(sim)
        results[cond_name]['success_rates'].append((sr1 + sr2) / 2)
        
        es = compute_effect_size(sim)
        results[cond_name]['effect_sizes'].append(es)
        
        idiff, i_self, i_others = compute_idiff(sim)
        results[cond_name]['idiffs'].append(idiff)
        results[cond_name]['i_selfs'].append(i_self)
        results[cond_name]['i_others'].append(i_others)
    
    print(f'{cond_name}:')
    print(f'  Mean success rate: {np.mean(results[cond_name]["success_rates"]):.1f}% ± {np.std(results[cond_name]["success_rates"]):.1f}')
    print(f'  Mean effect size:  {np.mean(results[cond_name]["effect_sizes"]):.2f} ± {np.std(results[cond_name]["effect_sizes"]):.2f}')
    print(f'  Mean Idiff:        {np.mean(results[cond_name]["idiffs"]):.4f} ± {np.std(results[cond_name]["idiffs"]):.4f}')
    print()

## 3. Panel A — Example similarity matrices

Pick one representative session pair (e.g. session 0 vs session 1) and show the 4 similarity matrices side-by-side.

In [ ]:
# Pick first session pair as representative
example_pair = session_pairs[0]
print(f'Example pair: session {example_pair[0]} vs session {example_pair[1]}')

fig, axes = plt.subplots(1, 4, figsize=(20, 5), sharey=True)
cmap = 'YlGnBu_r'

all_sims = [results[c]['sim_matrices'][example_pair] for c in conditions]
vmin = min(np.min(s) for s in all_sims)
vmax = max(np.max(s) for s in all_sims)

for ax, (cond_name, cond_results) in zip(axes, results.items()):
    sim = cond_results['sim_matrices'][example_pair]
    sr1, sr2 = compute_success_rate(sim)
    es = compute_effect_size(sim)
    
    im = ax.imshow(sim, cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_title(cond_name, fontsize=18)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.text(0.5, -0.07, 'Retest', transform=ax.transAxes, ha='center', fontsize=16)
    ax.text(0.5, -0.18, f'SR = {(sr1+sr2)/2:.0f}%  |  ES = {es:.2f}',
            transform=ax.transAxes, ha='center', fontsize=14,
            weight='bold' if 'Scaffold' in cond_name else 'normal')

axes[0].text(-0.07, 0.5, 'Test', va='center', rotation=90, fontsize=16, transform=axes[0].transAxes)

plt.subplots_adjust(right=0.9, wspace=0.05, bottom=0.2)
cbar_ax = fig.add_axes([0.91, 0.2, 0.015, 0.65])
fig.colorbar(im, cax=cbar_ax)

fig.suptitle(f'MSC — Session {example_pair[0]} vs {example_pair[1]}', fontsize=20, y=1.02)
plt.savefig(os.path.join(figure_dir, 'MSC_panel_A_example.png'), dpi=300, bbox_inches='tight')
plt.show()

## 4. Panel B — Success rate & effect size across all session pairs

In [ ]:
# Build long-form DataFrame for plotting
rows = []
for cond_name in conditions:
    for i, (s_test, s_retest) in enumerate(session_pairs):
        rows.append({
            'Condition': cond_name,
            'Session Pair': f'{s_test}-{s_retest}',
            'Success Rate': results[cond_name]['success_rates'][i],
            'Effect Size': results[cond_name]['effect_sizes'][i],
            'Idiff': results[cond_name]['idiffs'][i],
            'I_self': results[cond_name]['i_selfs'][i],
            'I_others': results[cond_name]['i_others'][i]
        })

df_results = pd.DataFrame(rows)
df_results.head()

In [ ]:
# ── Summary stats ──
summary = df_results.groupby('Condition').agg(
    SR_mean=('Success Rate', 'mean'),
    SR_std=('Success Rate', 'std'),
    ES_mean=('Effect Size', 'mean'),
    ES_std=('Effect Size', 'std'),
    Idiff_mean=('Idiff', 'mean'),
    Idiff_std=('Idiff', 'std'),
).round(3)

# Reorder
order = ['FC full', 'FC th=25%', 'FC th=1%', 'Scaffold']
summary = summary.loc[order]
summary

In [ ]:
# ── Box/swarm plot of success rates & effect sizes across 45 session pairs ──
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

palette = {'FC full': '#4C72B0', 'FC th=25%': '#55A868', 'FC th=1%': '#C44E52', 'Scaffold': '#DD8452'}

# Success Rate
sb.boxplot(data=df_results, x='Condition', y='Success Rate', order=order,
           palette=palette, ax=ax1, showfliers=False, width=0.5)
sb.stripplot(data=df_results, x='Condition', y='Success Rate', order=order,
             palette=palette, ax=ax1, alpha=0.5, size=5, jitter=0.15)
ax1.set_ylabel('Success Rate (%)', fontsize=14)
ax1.set_xlabel('')
ax1.set_title('Identification Accuracy', fontsize=16)
ax1.tick_params(axis='x', rotation=15)

# Effect Size
sb.boxplot(data=df_results, x='Condition', y='Effect Size', order=order,
           palette=palette, ax=ax2, showfliers=False, width=0.5)
sb.stripplot(data=df_results, x='Condition', y='Effect Size', order=order,
             palette=palette, ax=ax2, alpha=0.5, size=5, jitter=0.15)
ax2.set_ylabel("Cohen's d", fontsize=14)
ax2.set_xlabel('')
ax2.set_title('Effect Size (Intra vs Inter)', fontsize=16)
ax2.tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig(os.path.join(figure_dir, 'MSC_panel_B_boxplots.png'), dpi=300, bbox_inches='tight')
plt.show()

## 5. Intra vs Inter distribution (Panel B style)

In [ ]:
# For the representative pair, show intra/inter distributions
fig, axes = plt.subplots(1, 4, figsize=(21, 5.5), sharey=True)

blue_palette = sb.color_palette('Blues', n_colors=2)[::-1]
orange_palette = sb.color_palette('Oranges', n_colors=2)[::-1]

for idx, (cond_name, cond_results) in enumerate(results.items()):
    sim = cond_results['sim_matrices'][example_pair]
    intra = np.diag(sim)
    inter = sim[~np.eye(sim.shape[0], dtype=bool)]
    
    df_dist = pd.DataFrame({
        'type': ['Intra'] * len(intra) + ['Inter'] * len(inter),
        'value': np.concatenate([intra, inter])
    })
    
    pal = orange_palette if 'Scaffold' in cond_name else blue_palette
    
    sb.violinplot(x='type', y='value', data=df_dist, ax=axes[idx],
                  inner=None, hue='type', palette=pal, legend=False)
    sb.stripplot(x='type', y='value', data=df_dist, ax=axes[idx],
                 alpha=0.6, size=7, jitter=0.05, hue='type', palette=pal, legend=False)
    
    es = compute_effect_size(sim)
    axes[idx].set_title(f'{cond_name}\n(d = {es:.2f})', fontsize=16)
    axes[idx].set_xlabel('')
    if idx == 0:
        axes[idx].set_ylabel('Correlation', fontsize=14)
    else:
        axes[idx].set_ylabel('')

fig.suptitle(f'MSC — Intra vs Inter subject similarity (session {example_pair[0]} vs {example_pair[1]})', fontsize=18, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(figure_dir, 'MSC_intra_inter_violins.png'), dpi=300, bbox_inches='tight')
plt.show()

## 6. Multi-session average: mean similarity matrix across all 45 pairs

In [ ]:
# Average across all session pairs
fig, axes = plt.subplots(1, 4, figsize=(20, 5), sharey=True)
cmap = 'YlGnBu_r'

avg_sims = {}
for cond_name in conditions:
    sims = list(results[cond_name]['sim_matrices'].values())
    avg_sims[cond_name] = np.mean(sims, axis=0)

all_avgs = list(avg_sims.values())
vmin = min(np.min(s) for s in all_avgs)
vmax = max(np.max(s) for s in all_avgs)

for ax, (cond_name, avg_sim) in zip(axes, avg_sims.items()):
    sr1, sr2 = compute_success_rate(avg_sim)
    es = compute_effect_size(avg_sim)
    idiff, _, _ = compute_idiff(avg_sim)
    
    im = ax.imshow(avg_sim, cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_title(cond_name, fontsize=18)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.text(0.5, -0.07, 'Retest (avg)', transform=ax.transAxes, ha='center', fontsize=14)
    ax.text(0.5, -0.18, f'SR = {(sr1+sr2)/2:.0f}%  |  d = {es:.2f}  |  Idiff = {idiff:.3f}',
            transform=ax.transAxes, ha='center', fontsize=13,
            weight='bold' if 'Scaffold' in cond_name else 'normal')

axes[0].text(-0.07, 0.5, 'Test (avg)', va='center', rotation=90, fontsize=16, transform=axes[0].transAxes)

plt.subplots_adjust(right=0.9, wspace=0.05, bottom=0.2)
cbar_ax = fig.add_axes([0.91, 0.2, 0.015, 0.65])
fig.colorbar(im, cax=cbar_ax)

fig.suptitle('MSC — Mean similarity across all 45 session pairs', fontsize=20, y=1.02)
plt.savefig(os.path.join(figure_dir, 'MSC_avg_similarity.png'), dpi=300, bbox_inches='tight')
plt.show()

## 7. Session-pair heatmap of success rate

In [ ]:
# Build 10×10 session-pair success rate matrices for each condition
fig, axes = plt.subplots(1, 4, figsize=(22, 5), sharey=True)

for ax, cond_name in zip(axes, conditions):
    sr_matrix = np.full((n_sess, n_sess), np.nan)
    for (s_test, s_retest), sr in zip(session_pairs, results[cond_name]['success_rates']):
        sr_matrix[s_test, s_retest] = sr
        sr_matrix[s_retest, s_test] = sr
    
    sb.heatmap(sr_matrix, ax=ax, annot=True, fmt='.0f', cmap='YlOrRd',
               vmin=0, vmax=100, cbar=False, square=True,
               xticklabels=range(n_sess), yticklabels=range(n_sess))
    ax.set_title(f'{cond_name}\nMean SR = {np.mean(results[cond_name]["success_rates"]):.1f}%', fontsize=15)
    ax.set_xlabel('Session (retest)', fontsize=12)
    if ax == axes[0]:
        ax.set_ylabel('Session (test)', fontsize=12)

fig.suptitle('MSC — Success Rate per session pair', fontsize=18, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(figure_dir, 'MSC_session_pair_heatmap.png'), dpi=300, bbox_inches='tight')
plt.show()

## 8. Summary Table

In [ ]:
# Final summary
print('='*70)
print(f'{"Condition":15s} {"SR (%)":>12s} {"ES (d)":>12s} {"Idiff":>12s}')
print('='*70)
for cond in order:
    sr = results[cond]['success_rates']
    es = results[cond]['effect_sizes']
    idiff = results[cond]['idiffs']
    print(f'{cond:15s} {np.mean(sr):6.1f}±{np.std(sr):4.1f} {np.mean(es):6.2f}±{np.std(es):4.2f} {np.mean(idiff):6.4f}±{np.std(idiff):6.4f}')
print('='*70)